# LangChain + MCP 입문 가이드

## 이 노트북은 무엇을 배우나요?

```
┌─────────────────────────────────────────────────────────┐
│                                                         │
│   여러분의 코드          MCP 서버          OpenAI        │
│   (LangChain)    ←→   (도구 모음)   ←→   (GPT 모델)    │
│                                                         │
│   "15 곱하기 7"    →   multiply(15,7)  →  "105입니다"   │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

### 핵심 개념 3가지

**① MCP 서버** = AI가 사용할 수 있는 도구 창고
- 계산기, 파일 읽기, 날씨 조회 같은 기능들을 모아둔 곳
- 우리가 직접 만들거나 다른 사람이 만든 것을 사용할 수 있어요

**② LangChain** = AI 앱을 만드는 프레임워크
- MCP 서버의 도구를 AI 모델에 연결해줘요

**③ Agent** = 도구를 사용해서 문제를 스스로 해결하는 AI
- 질문을 받으면 → 어떤 도구를 쓸지 판단 → 도구 실행 → 결과 정리

---

### 학습 순서
```
STEP 1. 패키지 설치 & API 키 설정
STEP 2. MCP 서버 만들기 (도구 창고 만들기)
STEP 3. MCP 서버에 뭐가 있는지 확인하기
STEP 4. Agent 만들어서 질문하기
STEP 5. 대화 이어가기 (멀티턴)
STEP 6. 여러 MCP 서버 동시에 사용하기
```

---
## STEP 1. 패키지 설치 및 API 키 설정

### 필요한 패키지 설명
| 패키지 | 역할 |
|---|---|
| `mcp` | MCP 서버를 만들고 연결하는 핵심 라이브러리 |
| `langchain-mcp-adapters` | MCP 도구를 LangChain에서 쓸 수 있게 변환 |
| `langchain-openai` | OpenAI GPT 모델을 LangChain에서 사용 |
| `langchain` | `create_agent` 로 Agent 를 만드는 라이브러리 |
| `langgraph` | Agent 실행 그래프를 구성하는 라이브러리 |

In [ ]:
# 필요한 패키지를 한 번에 설치합니다
# uv add mcp langchain-mcp-adapters 

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 값은 출력하지 않고 설정 여부만 확인합니다 (노트북 출력에 키가 저장되는 것을 방지)
if not OPENAI_API_KEY:
    raise RuntimeError(".env 파일에 OPENAI_API_KEY 가 없습니다.")

print("✅ API 키 설정 완료")

✅ API 키 설정 완료


---
## STEP 2. MCP 서버 만들기

MCP 서버는 AI가 사용할 수 있는 **도구(Tool)들을 담은 서버**입니다.

아래에서는 간단한 **계산기 MCP 서버**를 만들어봅니다.

```
[ MCP 서버 구조 ]

@mcp.tool()       ← 이 데코레이터를 붙이면 AI가 사용할 수 있는 도구가 됩니다
def add(a, b):    ← 일반 Python 함수처럼 작성
    return a + b  ← 결과를 반환
```

> **💡 핵심**: 함수에 `@mcp.tool()`만 붙이면 끝! AI가 자동으로 이 함수를 도구로 인식합니다.

In [2]:
# math_server.py 파일로 저장합니다
# (별도 프로세스로 실행되어야 하기 때문에 파일로 만듭니다)

math_server_code = '''
import math
from mcp.server.fastmcp import FastMCP

# 입력 절댓값 상한 (비정상적으로 큰 값으로 계산이 깨지는 것을 방지)
LIMIT = 1e12

# MCP 서버 생성 (이름을 붙여줍니다)
mcp = FastMCP("계산기 서버")


def _check(*values: float) -> None:
    """입력이 유한한 수이고 허용 범위 안인지 검증합니다."""
    for v in values:
        if not math.isfinite(v) or abs(v) > LIMIT:
            raise ValueError(f"허용 범위(±{LIMIT:g})를 벗어난 값입니다")


# ---- 도구(Tool) 정의 ----
# @mcp.tool() 데코레이터를 붙이면 AI가 사용할 수 있는 도구가 됩니다
# 함수의 docstring(""" """)이 AI에게 전달되는 도구 설명입니다!

@mcp.tool()
def add(a: float, b: float) -> float:
    """두 숫자를 더합니다. 예: add(3, 5) → 8"""
    _check(a, b)
    return a + b

@mcp.tool()
def subtract(a: float, b: float) -> float:
    """두 숫자를 뺍니다. 예: subtract(10, 3) → 7"""
    _check(a, b)
    return a - b

@mcp.tool()
def multiply(a: float, b: float) -> float:
    """두 숫자를 곱합니다. 예: multiply(4, 5) → 20"""
    _check(a, b)
    return a * b

@mcp.tool()
def divide(a: float, b: float) -> float:
    """두 숫자를 나눕니다. 예: divide(10, 2) → 5. 단, b가 0이면 에러!"""
    _check(a, b)
    if b == 0:
        raise ValueError("0으로 나눌 수 없습니다!")
    result = a / b
    if not math.isfinite(result):   # 아주 작은 b 로 나누면 결과가 무한대가 될 수 있음
        raise ValueError("결과가 너무 커서 계산할 수 없습니다")
    return result

# 서버 실행 (stdio 방식 = 표준 입출력으로 통신)
if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

# 파일로 저장
with open("math_server.py", "w", encoding="utf-8") as f:
    f.write(math_server_code)

print("✅ math_server.py 생성 완료!")
print()
print("📦 이 서버가 제공하는 도구:")
print("  - add(a, b)      : 더하기")
print("  - subtract(a, b) : 빼기")
print("  - multiply(a, b) : 곱하기")
print("  - divide(a, b)   : 나누기")
print()
print("🛡️ 입력값은 유한한 수이고 ±1e12 이내일 때만 계산합니다.")

✅ math_server.py 생성 완료!

📦 이 서버가 제공하는 도구:
  - add(a, b)      : 더하기
  - subtract(a, b) : 빼기
  - multiply(a, b) : 곱하기
  - divide(a, b)   : 나누기

🛡️ 입력값은 유한한 수이고 ±1e12 이내일 때만 계산합니다.


---
## STEP 3. MCP 서버에 연결하고 도구 목록 확인하기

서버를 만들었으니, 실제로 연결해서 어떤 도구들이 있는지 확인해봅시다.

```
[ 연결 흐름 ]

 내 코드
   │
   ├─ stdio_client()  ← 서버 프로세스를 실행하고 파이프로 연결
   │
   └─ ClientSession() ← 연결된 서버와 대화하는 세션
         │
         ├─ session.list_tools()    → 도구 목록 조회
         └─ load_mcp_tools(session) → LangChain 형식으로 변환
```

> **💡 왜 async/await?**
> MCP 통신은 네트워크처럼 기다림이 발생해요. `async/await`는 기다리는 동안 다른 작업을 할 수 있게 해줍니다.

> **🪟 Windows + Jupyter 주의사항**
> Jupyter의 `sys.stderr`는 가상 스트림이라 MCP 서버 프로세스에 직접 연결할 수 없어요.
> 그래서 `errlog`에 **실제 파일**(`mcp_server.log`)을 전달합니다. 이 옵션이 없으면 `UnsupportedOperation: fileno` 에러가 발생합니다!
> `subprocess.DEVNULL`로 버리는 방법도 있지만, 서버가 실패해도 **원인이 보이지 않아** 권장하지 않습니다. 연결이 실패하면 `mcp_server.log`를 먼저 확인하세요.

> **🔒 안전하게 쓰기 위한 두 가지**
> ① 서버는 `sys.executable`(지금 노트북을 실행 중인 파이썬)으로 실행합니다. `"python"`만 쓰면 PATH 에 따라 다른 파이썬이 실행될 수 있어요.
> ② `select_allowed()`로 **허용한 도구만** Agent 에 넘깁니다. 서버가 나중에 도구를 추가하거나 바뀌어도 자동으로 노출되지 않아요.

In [3]:
import sys
from pathlib import Path
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

# 서버 오류 메시지를 기록할 로그 파일 (Jupyter 호환 + 실패 원인 확인용)
ERRLOG_PATH = Path("mcp_server.log")

# Agent 에게 허용할 도구 이름 (목록에 없는 도구는 서버가 제공해도 사용하지 않음)
ALLOWED_TOOLS = {"add", "subtract", "multiply", "divide"}


def select_allowed(tools, allowed=ALLOWED_TOOLS):
    """허용 목록에 있는 도구만 골라 반환합니다. 목록에 없는 도구가 있으면 알려줍니다."""
    selected = [t for t in tools if t.name in allowed]
    excluded = sorted(t.name for t in tools if t.name not in allowed)
    if excluded:
        print(f"⚠️ 허용 목록에 없어 제외한 도구: {excluded}")
    return selected


def make_server_params(script: str) -> StdioServerParameters:
    """현재 노트북을 실행 중인 파이썬(sys.executable)으로 서버 스크립트를 실행하는 설정을 만듭니다."""
    return StdioServerParameters(command=sys.executable, args=[str(Path(script).resolve())])


# MCP 서버 실행 정보 설정
server_params = make_server_params("math_server.py")


async def check_available_tools():
    """MCP 서버에 연결해서 사용 가능한 도구를 확인합니다"""

    # errlog 에 실제 파일을 전달 → Windows + Jupyter 에서도 동작하고, 서버 오류가 파일에 남음
    with open(ERRLOG_PATH, "a", encoding="utf-8") as errlog:
        async with stdio_client(server_params, errlog=errlog) as (read, write):
            # 연결된 서버와 대화할 세션을 만듭니다
            async with ClientSession(read, write) as session:

                # 세션 초기화 (악수하기 - 서로를 인식하는 과정)
                await session.initialize()

                # ① 원시 MCP 도구 목록 조회 (어떤 도구들이 있는지 확인)
                tools_info = await session.list_tools()
                print("📋 MCP 서버의 원시 도구 목록:")
                for tool in tools_info.tools:
                    print(f"  🔧 {tool.name}: {tool.description}")

                print()

                # ② LangChain 형식으로 변환 + 허용한 도구만 선택
                langchain_tools = select_allowed(await load_mcp_tools(session))
                print("🔄 LangChain 형식으로 변환된 도구들:")
                for tool in langchain_tools:
                    print(f"  ✅ {tool.name}: {tool.description}")

                print()
                print(f"총 {len(langchain_tools)}개의 도구를 Agent에서 사용할 수 있습니다!")

# Jupyter에서는 await를 직접 사용할 수 있습니다
await check_available_tools()

c:\myclass\ai_langchain\mylangchin-uv-project\.venv\Lib\site-packages\langgraph\checkpoint\base\__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


📋 MCP 서버의 원시 도구 목록:
  🔧 add: 두 숫자를 더합니다. 예: add(3, 5) → 8
  🔧 subtract: 두 숫자를 뺍니다. 예: subtract(10, 3) → 7
  🔧 multiply: 두 숫자를 곱합니다. 예: multiply(4, 5) → 20
  🔧 divide: 두 숫자를 나눕니다. 예: divide(10, 2) → 5. 단, b가 0이면 에러!

🔄 LangChain 형식으로 변환된 도구들:
  ✅ add: 두 숫자를 더합니다. 예: add(3, 5) → 8
  ✅ subtract: 두 숫자를 뺍니다. 예: subtract(10, 3) → 7
  ✅ multiply: 두 숫자를 곱합니다. 예: multiply(4, 5) → 20
  ✅ divide: 두 숫자를 나눕니다. 예: divide(10, 2) → 5. 단, b가 0이면 에러!

총 4개의 도구를 Agent에서 사용할 수 있습니다!


---
## STEP 4. Agent 만들어서 질문하기

이제 GPT 모델 + MCP 도구를 합쳐서 **Agent**를 만들어봅시다!

```
[ Agent 동작 방식 - ReAct 패턴 ]

 사용자: "15 곱하기 7을 계산하고 23을 빼줘"
      │
      ▼
 🤖 GPT가 생각: "곱하기 먼저 해야겠다 → multiply 도구 사용!"
      │
      ▼
 🔧 도구 실행: multiply(15, 7) → 105
      │
      ▼
 🤖 GPT가 생각: "105에서 23을 빼야한다 → subtract 도구 사용!"
      │
      ▼
 🔧 도구 실행: subtract(105, 23) → 82
      │
      ▼
 💬 최종 답변: "15 × 7 = 105, 105 - 23 = 82입니다!"
```

In [4]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent   # langgraph.prebuilt.create_react_agent 는 deprecated

# GPT 모델 초기화
# temperature=0 → 일관된 답변 (창의성 없이 정확하게)
# model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind(parallel_tool_calls=False)

# server_params, ERRLOG_PATH, select_allowed 는 STEP 3 셀에서 정의한 것을 그대로 사용합니다

async def ask_agent(question: str):
    """Agent에게 질문하고 결과를 보기 좋게 출력합니다"""

    # errlog 에 실제 파일을 전달 (Windows + Jupyter 호환, 서버 오류는 파일에 기록)
    with open(ERRLOG_PATH, "a", encoding="utf-8") as errlog:
        async with stdio_client(server_params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                # MCP 도구를 LangChain 형식으로 로드하고, 허용한 도구만 선택
                tools = select_allowed(await load_mcp_tools(session))

                # Agent 생성 (GPT 모델 + MCP 도구들)
                agent = create_agent(model, tools)

                print(f"❓ 질문: {question}")
                print("-" * 50)

                # Agent 실행
                # messages 형식으로 질문을 전달합니다
                result = await agent.ainvoke({
                    "messages": [{"role": "user", "content": question}]
                })

                # 중간 과정 출력 (Agent가 어떤 생각을 했는지)
                print("🔍 Agent의 처리 과정:")
                for msg in result["messages"]:
                    msg_type = type(msg).__name__

                    if msg_type == "HumanMessage":
                        print(f"  👤 사용자 입력: {msg.content}")
                    elif msg_type == "AIMessage":
                        if msg.tool_calls:  # 도구를 호출하는 경우
                            for tc in msg.tool_calls:
                                print(f"  🤖 AI 결정: {tc['name']} 도구 사용 → 인자: {tc['args']}")
                        else:  # 최종 답변
                            print(f"  💬 최종 답변: {msg.content}")
                    elif msg_type == "ToolMessage":
                        print(f"  🔧 도구 결과: {msg.content}")

                return result["messages"][-1].content

# 테스트 실행!
#answer = await ask_agent("15 곱하기 7을 계산하고, 계산한 결과값에서 23을 빼줘")
answer = await ask_agent("먼저 15 곱하기 7을 계산해. 그 다음, 앞에서 나온 곱셈 결과에서 23을 빼줘. 각 단계를 순서대로 실행해.")

❓ 질문: 먼저 15 곱하기 7을 계산해. 그 다음, 앞에서 나온 곱셈 결과에서 23을 빼줘. 각 단계를 순서대로 실행해.
--------------------------------------------------
🔍 Agent의 처리 과정:
  👤 사용자 입력: 먼저 15 곱하기 7을 계산해. 그 다음, 앞에서 나온 곱셈 결과에서 23을 빼줘. 각 단계를 순서대로 실행해.
  🤖 AI 결정: multiply 도구 사용 → 인자: {'a': 15, 'b': 7}
  🔧 도구 결과: [{'type': 'text', 'text': '105.0', 'id': 'lc_20dde115-b954-4537-9e4a-5c2d791f59f4'}]
  🤖 AI 결정: subtract 도구 사용 → 인자: {'a': 105, 'b': 23}
  🔧 도구 결과: [{'type': 'text', 'text': '82.0', 'id': 'lc_7f498eee-3779-4615-87c3-7f4224757bd4'}]
  💬 최종 답변: 15 곱하기 7은 105입니다. 그 결과에서 23을 빼면 82가 됩니다.


---
## STEP 5. 대화 이어가기 (멀티턴 대화)

이전 대화를 기억하면서 계속 이야기하는 방법입니다.

```
[ 멀티턴 동작 방식 ]

  messages = []  ← 대화 기록을 저장하는 리스트

  1번째 질문: "12 곱하기 8은?"
  messages = [
    {user: "12 곱하기 8은?"},
    {assistant: "96입니다"}    ← 대화 기록 추가
  ]

  2번째 질문: "거기서 30을 빼면?"  ← "거기서"가 96을 가리킴!
  messages = [
    {user: "12 곱하기 8은?"},
    {assistant: "96입니다"},
    {user: "거기서 30을 빼면?"},    ← 이전 대화가 있어서 문맥 파악 가능
    {assistant: "66입니다"}
  ]
```

> **💡 포인트**: 매번 `messages` 전체를 Agent에 전달해야 이전 대화를 기억합니다!

In [5]:
async def multi_turn_chat():
    """대화 기록을 유지하며 여러 번 대화합니다"""

    # errlog 에 실제 파일을 전달 (Windows + Jupyter 호환, 서버 오류는 파일에 기록)
    with open(ERRLOG_PATH, "a", encoding="utf-8") as errlog:
        async with stdio_client(server_params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = select_allowed(await load_mcp_tools(session))
                agent = create_agent(model, tools)

                # 📝 대화 기록을 저장할 리스트
                messages = []

                # 연속으로 할 질문들
                questions = [
                    "12 곱하기 8은 얼마야?",
                    "방금 나온 결과에서 30을 빼면?",   # '방금 나온 결과' = 96
                    "그 결과를 2로 나눠줘",             # '그 결과' = 66
                ]

                print("=" * 50)
                print("💬 멀티턴 대화 시작")
                print("=" * 50)

                for i, question in enumerate(questions, 1):
                    print(f"\n[{i}번째 대화]")
                    print(f"👤 나: {question}")

                    # 새 질문을 대화 기록에 추가
                    messages.append({"role": "user", "content": question})

                    # 전체 대화 기록을 Agent에 전달 (이전 문맥 포함!)
                    result = await agent.ainvoke({"messages": messages})

                    # AI 응답 추출
                    ai_response = result["messages"][-1].content

                    # AI 응답도 대화 기록에 추가 (다음 번 질문에서 참조)
                    messages.append({"role": "assistant", "content": ai_response})

                    print(f"🤖 AI: {ai_response}")
                    print(f"   (현재 대화 기록 수: {len(messages)}개)")

                print("\n" + "=" * 50)
                print("✅ 대화 완료")

await multi_turn_chat()

💬 멀티턴 대화 시작

[1번째 대화]
👤 나: 12 곱하기 8은 얼마야?
🤖 AI: 12 곱하기 8은 96입니다.
   (현재 대화 기록 수: 2개)

[2번째 대화]
👤 나: 방금 나온 결과에서 30을 빼면?
🤖 AI: 96에서 30을 빼면 66이 됩니다.
   (현재 대화 기록 수: 4개)

[3번째 대화]
👤 나: 그 결과를 2로 나눠줘
🤖 AI: 66을 2로 나누면 33이 됩니다.
   (현재 대화 기록 수: 6개)

✅ 대화 완료


---
## STEP 6. 여러 MCP 서버 동시에 사용하기

계산기 서버(`math_server.py`)와 텍스트 서버(`text_server.py`)를 **한 Agent 에 함께** 연결합니다.

```
 Agent ──┬─ 세션 1 ── math_server.py  (add, subtract, multiply, divide)
         └─ 세션 2 ── text_server.py  (to_uppercase, count_words, reverse_text)
```

### 방법: 서버마다 세션을 열고 도구를 합칩니다
- `AsyncExitStack` 으로 여러 세션을 한 번에 열고, 함수가 끝나면 한꺼번에 닫습니다.
- 서로 다른 서버의 **도구 이름이 겹치면 안 됩니다.** (Agent 가 구분하지 못함)

> **ℹ️ `MultiServerMCPClient` 를 쓰지 않는 이유 (langchain-mcp-adapters 0.2.2 에서 확인)**
> - `async with MultiServerMCPClient(...) as client:` 형태는 지원되지 않아 `NotImplementedError` 가 납니다. (`client = MultiServerMCPClient({...})` 후 `await client.get_tools()` 로 사용)
> - 서버 설정에 `errlog` 키가 없어서 넣으면 `TypeError` 가 납니다. 서버 오류를 로그 파일에 남기려면 이 노트북처럼 `stdio_client(..., errlog=파일)` 로 직접 연결해야 합니다.

In [6]:
from contextlib import AsyncExitStack

# text_server.py 파일로 저장합니다 (math_server.py 와 같은 방식)
text_server_code = '''
from mcp.server.fastmcp import FastMCP

# 입력 길이 상한 (과도하게 긴 입력으로 인한 자원 낭비를 방지)
MAX_TEXT_CHARS = 10_000

mcp = FastMCP("텍스트 처리 서버")


def _check_len(text: str) -> None:
    """입력 길이가 상한을 넘으면 오류를 발생시킵니다."""
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"텍스트가 너무 깁니다 ({len(text):,}자 > {MAX_TEXT_CHARS:,}자)")


@mcp.tool()
def to_uppercase(text: str) -> str:
    """텍스트를 대문자로 변환합니다. 예: to_uppercase('hello') → 'HELLO'"""
    _check_len(text)
    return text.upper()

@mcp.tool()
def count_words(text: str) -> int:
    """텍스트의 단어 수를 세어 반환합니다. 예: count_words('hello world') → 2"""
    _check_len(text)
    return len(text.split())

@mcp.tool()
def reverse_text(text: str) -> str:
    """텍스트를 거꾸로 뒤집습니다. 예: reverse_text('hello') → 'olleh'"""
    _check_len(text)
    return text[::-1]

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("text_server.py", "w", encoding="utf-8") as f:
    f.write(text_server_code)
print("✅ text_server.py 생성 완료!")

# 연결할 서버 목록 (서버 이름 → 스크립트)
SERVER_SCRIPTS = {"math": "math_server.py", "text": "text_server.py"}

# 두 서버에서 허용할 도구 이름
MULTI_ALLOWED_TOOLS = ALLOWED_TOOLS | {"to_uppercase", "count_words", "reverse_text"}


async def connect_servers(stack: AsyncExitStack):
    """모든 서버에 연결하고, 허용한 도구를 합쳐서 반환합니다. (세션은 stack 이 닫힐 때 함께 닫힘)"""
    errlog = stack.enter_context(open(ERRLOG_PATH, "a", encoding="utf-8"))
    all_tools = []
    for script in SERVER_SCRIPTS.values():
        read, write = await stack.enter_async_context(stdio_client(make_server_params(script), errlog=errlog))
        session = await stack.enter_async_context(ClientSession(read, write))
        await session.initialize()
        all_tools += await load_mcp_tools(session)

    # 도구 이름 중복 검사
    names = [t.name for t in all_tools]
    duplicated = sorted({n for n in names if names.count(n) > 1})
    if duplicated:
        raise ValueError(f"서버 간 도구 이름이 겹칩니다: {duplicated}")
    return select_allowed(all_tools, MULTI_ALLOWED_TOOLS)


async def show_multi_tools():
    """두 서버의 도구 목록을 확인합니다 (LLM 호출 없음)"""
    async with AsyncExitStack() as stack:
        tools = await connect_servers(stack)
        for tool in tools:
            print(f"  ✅ {tool.name}: {tool.description}")
        print(f"\n총 {len(tools)}개의 도구를 한 Agent 에서 사용할 수 있습니다!")

await show_multi_tools()

✅ text_server.py 생성 완료!
  ✅ add: 두 숫자를 더합니다. 예: add(3, 5) → 8
  ✅ subtract: 두 숫자를 뺍니다. 예: subtract(10, 3) → 7
  ✅ multiply: 두 숫자를 곱합니다. 예: multiply(4, 5) → 20
  ✅ divide: 두 숫자를 나눕니다. 예: divide(10, 2) → 5. 단, b가 0이면 에러!
  ✅ to_uppercase: 텍스트를 대문자로 변환합니다. 예: to_uppercase('hello') → 'HELLO'
  ✅ count_words: 텍스트의 단어 수를 세어 반환합니다. 예: count_words('hello world') → 2
  ✅ reverse_text: 텍스트를 거꾸로 뒤집습니다. 예: reverse_text('hello') → 'olleh'

총 7개의 도구를 한 Agent 에서 사용할 수 있습니다!


In [7]:
async def ask_multi(question: str):
    """두 MCP 서버의 도구를 모두 가진 Agent 에게 질문합니다"""
    async with AsyncExitStack() as stack:
        tools = await connect_servers(stack)
        agent = create_agent(model, tools)

        print(f"❓ 질문: {question}")
        print("-" * 50)
        result = await agent.ainvoke({"messages": [{"role": "user", "content": question}]})

        for msg in result["messages"]:
            if type(msg).__name__ == "AIMessage" and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  🤖 AI 결정: {tc['name']} 도구 사용 → 인자: {tc['args']}")
        print(f"  💬 최종 답변: {result['messages'][-1].content}")
        return result["messages"][-1].content

# 텍스트 서버(count_words)와 계산기 서버(multiply)를 함께 써야 풀 수 있는 질문
answer = await ask_multi("'hello world from mcp' 의 단어 수를 세고, 그 숫자에 10을 곱해줘")

❓ 질문: 'hello world from mcp' 의 단어 수를 세고, 그 숫자에 10을 곱해줘
--------------------------------------------------
  🤖 AI 결정: count_words 도구 사용 → 인자: {'text': 'hello world from mcp'}
  🤖 AI 결정: multiply 도구 사용 → 인자: {'a': 4, 'b': 10}
  💬 최종 답변: 'hello world from mcp'의 단어 수는 4개이며, 이 숫자에 10을 곱하면 40이 됩니다.


---
##  전체 정리

### 배운 내용 요약

```
┌─────────────────────────────────────────────────────────┐
│                   LangChain + MCP 흐름                  │
│                                                         │
│  1. MCP 서버 만들기                                      │
│     FastMCP + @mcp.tool() 데코레이터                     │
│                                                         │
│  2. 연결 & 도구 로드                                     │
│     stdio_client → ClientSession → load_mcp_tools()     │
│     → select_allowed() 로 허용한 도구만 선택              │
│                                                         │
│  3. Agent 만들기                                         │
│     create_agent(모델, 도구들)                            │
│                                                         │
│  4. 질문하기                                             │
│     agent.ainvoke({"messages": [...]})                  │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

### 🪟 Windows + Jupyter 필수 패턴

```python
import sys
from pathlib import Path

# errlog 에는 fileno 가 있는 "실제 파일"을 전달합니다 (Jupyter 의 가상 stderr 는 사용 불가)
params = StdioServerParameters(command=sys.executable, args=[str(Path("math_server.py").resolve())])
with open("mcp_server.log", "a", encoding="utf-8") as errlog:
    async with stdio_client(params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            ...
```

- 여러 서버는 STEP 6 처럼 서버마다 세션을 열고 `AsyncExitStack` 으로 묶습니다.
- `MultiServerMCPClient` (langchain-mcp-adapters 0.2.x): `async with` 로는 쓸 수 없고, 서버 설정에 `errlog` 키도 없습니다.

### 🔒 안전하게 쓰기 체크리스트
- 서버는 `sys.executable` + 절대 경로로 실행하기 (PATH 의 `python` 에 의존하지 않기)
- `select_allowed()` 로 **허용한 도구만** Agent 에 넘기기 (서버가 도구를 추가해도 자동 노출 금지)
- 서버 도구 안에서 **입력 범위·길이 검증**하기 (이 노트북의 `_check`, `_check_len`)
- `env` 를 지정할 때 `os.environ` 을 통째로 넘기지 말고 **필요한 변수만** 넘기기 (API 키 유출 방지)
- API 키는 **값을 출력하지 않기** (노트북 출력에 저장됨)
- 외부에서 받은 MCP 서버는 버전을 고정하고 코드를 검토한 뒤 사용하기 → 자세한 실습: `owasp/before_after/04_ASI02_MCP_도구권한.ipynb`

### 다음 단계로 나아가기
- 🔍 [LangChain MCP Adapters 공식 문서](https://github.com/langchain-ai/langchain-mcp-adapters)
- 🌐 [MCP 공식 서버 목록](https://github.com/modelcontextprotocol/servers) - 파일시스템, GitHub, Slack 등 다양한 서버
- 📚 [LangGraph 문서](https://langchain-ai.github.io/langgraph/) - 더 복잡한 Agent 만들기